# Data Cleaning and Feature Engineering


In [1]:
import os
project_root = os.path.abspath('..')
os.chdir(project_root)
print("Working directory:", os.getcwd())

Working directory: /Users/ac/Desktop/postharvest-iq


In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to MySQL successfully")

Connected to MySQL successfully


In [3]:
# Load cereal prices — Northern markets + Southern reference markets
prices = pd.read_sql("""
    SELECT p.date, p.market, p.market_id, p.commodity,
           p.price, p.pricetype, p.currency,
           m.latitude, m.longitude, m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
""", engine)

prices['date'] = pd.to_datetime(prices['date'])
prices['year'] = prices['date'].dt.year
prices['month'] = prices['date'].dt.month

print(f"Total records: {len(prices)}")
print(f"Markets: {prices['market'].unique()}")
print(f"Commodities: {prices['commodity'].unique()}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
print(f"Missing values: {prices.isnull().sum().sum()}")
print()
print("Records per market per commodity:")
print(prices.groupby(['market','commodity']).size().to_string())

Total records: 1770
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2006-01-15 00:00:00 to 2023-07-15 00:00:00
Missing values: 0

Records per market per commodity:
market    commodity
Bolga     Maize        108
          Millet        78
          Sorghum       71
Kumasi    Maize        206
          Millet       188
          Sorghum      127
Tamale    Maize        100
          Millet       129
          Sorghum       76
Techiman  Maize        163
          Millet       195
          Sorghum      118
Wa        Maize         67
          Millet        74
          Sorghum       70


In [4]:
# Load exchange rates — annual values only
fx = pd.read_sql("""
    SELECT year, value as exchange_rate
    FROM ghana_exchange_rates
    WHERE months = 'Annual value'
    ORDER BY year
""", engine)

fx['year'] = fx['year'].astype(int)

# Ghana redenominated the cedi in 2007 — exclude the transition year and earlier
# (2005-2007 rows contain both old and new denomination values; from 2008 only new)
fx_clean = fx[fx['year'] >= 2008].copy()
fx_clean = fx_clean.groupby('year')['exchange_rate'].mean().reset_index()

print(f"Exchange rate records: {len(fx_clean)}")
print(fx_clean)

# Merge exchange rates on year; fx_clean has one row per year after grouping
prices = prices.merge(fx_clean, on='year', how='left')

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing exchange rates: {prices['exchange_rate'].isnull().sum()}")

Exchange rate records: 18
    year  exchange_rate
0   2008       1.052275
1   2009       1.404967
2   2010       1.429983
3   2011       1.520625
4   2012       1.824867
5   2013       1.981350
6   2014       2.896575
7   2015       3.714642
8   2016       3.909817
9   2017       4.350533
10  2018       4.585325
11  2019       5.217367
12  2020       5.595708
13  2021       5.805700
14  2022       8.272400
15  2023      11.020409
16  2024      14.181942
17  2025      12.527120

After merge: 1770 rows
Missing exchange rates: 214


In [5]:
# Load producer price index — the only FAO element with complete 2008-2023 coverage.
# LCU/USD/SLC price elements all missing 2012-2015 and 2023 in this dataset.
# Index (2014-2016=100) is dimensionless and tracks relative farm gate price changes.
producer = pd.read_sql("""
    SELECT year, item, value as producer_price_index
    FROM fao_producer_prices
    WHERE item IN ('Maize (corn)', 'Millet', 'Sorghum')
    AND months = 'Annual value'
    AND element = 'Producer Price Index (2014-2016 = 100)'
    ORDER BY item, year
""", engine)

producer['year'] = producer['year'].astype(int)
producer['item'] = producer['item'].replace('Maize (corn)', 'Maize')

producer_clean = producer[producer['year'] >= 2008].copy()
producer_clean = producer_clean.groupby(
    ['year', 'item']
)['producer_price_index'].mean().reset_index()
producer_clean = producer_clean.rename(columns={'item': 'commodity'})

print(f"Producer price index records: {len(producer_clean)}")
print(producer_clean.pivot(index='year', columns='commodity', values='producer_price_index').round(1))

prices = prices.merge(producer_clean, on=['year', 'commodity'], how='left')

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing producer index: {prices['producer_price_index'].isnull().sum()}")

Producer price index records: 54
commodity  Maize  Millet  Sorghum
year                             
2008        36.0    37.6     39.4
2009        41.4    46.1     47.9
2010        37.4    46.9     48.8
2011        49.8    51.7     55.6
2012        60.3    61.7     65.2
2013        74.2    76.3     78.8
2014        84.8    86.1     89.0
2015       101.6   102.8    101.3
2016       113.6   111.2    109.7
2017        85.2    78.3     72.1
2018        99.8   101.5    105.1
2019        80.9   100.5     97.4
2020       126.0   119.1    126.8
2021       185.4   190.4    192.4
2022       288.4   279.3    279.7
2023       377.2   365.0    351.5
2024       409.4   384.8    374.6
2025       441.6   475.6    455.2

After merge: 1770 rows
Missing producer index: 214


In [6]:
# Remove outliers per market per commodity using IQR on real (deflated) prices
# Deflating avoids misclassifying recent high-inflation observations as outliers
prices['real_price'] = prices['price'] / prices['exchange_rate']

n_before = len(prices)
print(f"Records before outlier removal: {n_before}")

bounds = (
    prices.groupby(['market', 'commodity'])['real_price']
    .agg(Q1=lambda x: x.quantile(0.25), Q3=lambda x: x.quantile(0.75))
    .reset_index()
)
bounds['IQR'] = bounds['Q3'] - bounds['Q1']
bounds['lower'] = bounds['Q1'] - 3 * bounds['IQR']
bounds['upper'] = bounds['Q3'] + 3 * bounds['IQR']

prices = (
    prices
    .merge(bounds[['market', 'commodity', 'lower', 'upper']], on=['market', 'commodity'])
    .loc[lambda d: (d['real_price'] >= d['lower']) & (d['real_price'] <= d['upper'])]
    .drop(columns=['lower', 'upper'])
    .reset_index(drop=True)
)

print(f"Records after outlier removal: {len(prices)}")
print(f"Records removed: {n_before - len(prices)}")
print()
print("Records per market per commodity after cleaning:")
print(prices.groupby(['market', 'commodity']).size().to_string())

Records before outlier removal: 1770
Records after outlier removal: 1555
Records removed: 215

Records per market per commodity after cleaning:
market    commodity
Bolga     Maize         98
          Millet        72
          Sorghum       62
Kumasi    Maize        182
          Millet       165
          Sorghum      109
Tamale    Maize         86
          Millet       113
          Sorghum       63
Techiman  Maize        146
          Millet       171
          Sorghum      102
Wa        Maize         59
          Millet        69
          Sorghum       58


In [7]:
# Add time features
prices['month_sin'] = np.sin(2 * np.pi * prices['month'] / 12)
prices['month_cos'] = np.cos(2 * np.pi * prices['month'] / 12)

print(f"Month sin range: {prices['month_sin'].min():.2f} to {prices['month_sin'].max():.2f}")
print(f"Month cos range: {prices['month_cos'].min():.2f} to {prices['month_cos'].max():.2f}")

Month sin range: -1.00 to 1.00
Month cos range: -1.00 to 1.00


In [8]:
final_columns = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'exchange_rate', 'producer_price_index',
    'month_sin', 'month_cos'
]

df_final = (
    prices[final_columns]
    .copy()
    .dropna()
    .sort_values(['market', 'commodity', 'date'])
    .reset_index(drop=True)
)

os.makedirs('data/processed', exist_ok=True)
df_final.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print(f"Total rows: {len(df_final)}")
print(f"Columns: {list(df_final.columns)}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Markets: {df_final['market'].unique()}")
print(f"Commodities: {df_final['commodity'].unique()}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")
print(f"CSV file: {os.path.exists('data/processed/cereals_merged_clean.csv')}")
print()
print("Records per market per commodity:")
print(df_final.groupby(['market', 'commodity']).size().to_string())

Total rows: 1555
Columns: ['date', 'market', 'commodity', 'price', 'latitude', 'longitude', 'exchange_rate', 'producer_price_index', 'month_sin', 'month_cos']
Missing values: 0
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2008-01-15 00:00:00 to 2023-07-15 00:00:00
CSV file: True

Records per market per commodity:
market    commodity
Bolga     Maize         98
          Millet        72
          Sorghum       62
Kumasi    Maize        182
          Millet       165
          Sorghum      109
Tamale    Maize         86
          Millet       113
          Sorghum       63
Techiman  Maize        146
          Millet       171
          Sorghum      102
Wa        Maize         59
          Millet        69
          Sorghum       58
